# 02 — Model Training & Evaluation
## NetGuard AI: ML-Powered Network Intrusion Detection

This notebook trains and evaluates all detection models:
1. Random Forest
2. XGBoost
3. Autoencoder (anomaly detection)
4. Ensemble (combined)

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from netguard.preprocessing.loader import load_nsl_kdd
from netguard.preprocessing.features import prepare_dataset
from netguard.models.random_forest import RFDetector
from netguard.models.xgboost_model import XGBDetector
from netguard.models.autoencoder import AEDetector
from netguard.models.ensemble import EnsembleDetector
from netguard.evaluation.metrics import (
    evaluate_binary, compare_models, plot_confusion_matrix, 
    plot_roc_curves, print_classification_report
)

os.makedirs('../models', exist_ok=True)
os.makedirs('../docs/figures', exist_ok=True)
print('Ready')

## 1. Load and Preprocess Data

In [ ]:
train_df = load_nsl_kdd('train')
test_df = load_nsl_kdd('test')

X_train, y_train, scaler, features = prepare_dataset(train_df, top_k_features=30)

# Prepare test with same features
X_test_full, y_test, _, _ = prepare_dataset(test_df, top_k_features=0)
common_features = [f for f in features if f in X_test_full.columns]
X_test = X_test_full[common_features]

# Re-align train to common features
X_train = X_train[common_features]
features = common_features

print(f'Train: {X_train.shape}')
print(f'Test: {X_test.shape}')
print(f'Features: {len(features)}')
print(f'Train attack ratio: {y_train.mean():.2%}')
print(f'Test attack ratio: {y_test.mean():.2%}')

## 2. Train Random Forest

In [ ]:
rf = RFDetector()
rf.train(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)
rf_results = evaluate_binary(y_test, rf_pred, rf_proba, 'Random Forest')

print_classification_report(y_test, rf_pred, 'Random Forest')
plot_confusion_matrix(y_test, rf_pred, labels=[0, 1], title='Random Forest', save_path='../docs/figures/confusion_matrix_rf.png')

rf.save('../models/rf_model.pkl')

## 3. Train XGBoost

In [ ]:
xgb = XGBDetector()
xgb.train(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)
xgb_results = evaluate_binary(y_test, xgb_pred, xgb_proba, 'XGBoost')

print_classification_report(y_test, xgb_pred, 'XGBoost')
plot_confusion_matrix(y_test, xgb_pred, labels=[0, 1], title='XGBoost', save_path='../docs/figures/confusion_matrix_xgb.png')

xgb.save('../models/xgb_model.pkl')

## 4. Train Autoencoder

In [ ]:
# Train only on normal traffic
X_train_normal = X_train[y_train == 0]
print(f'Training Autoencoder on {len(X_train_normal)} normal samples')

ae = AEDetector(encoding_dim=12, threshold_percentile=95)
ae.train(X_train_normal, epochs=50, batch_size=256)

ae_pred = ae.predict(X_test)
ae_scores = ae.predict_scores(X_test)
ae_results = evaluate_binary(y_test, ae_pred, ae_scores, 'Autoencoder')

print_classification_report(y_test, ae_pred, 'Autoencoder')
plot_confusion_matrix(y_test, ae_pred, labels=[0, 1], title='Autoencoder', save_path='../docs/figures/confusion_matrix_ae.png')

ae.save('../models/ae_model.pt')

## 5. Ensemble

In [ ]:
ensemble = EnsembleDetector(weights={'rf': 0.35, 'xgb': 0.35, 'ae': 0.30})
ensemble.add_model('rf', rf)
ensemble.add_model('xgb', xgb)
ensemble.add_model('ae', ae)

ens_pred = ensemble.predict(X_test)
ens_scores = ensemble.predict_scores(X_test)
ens_results = evaluate_binary(y_test, ens_pred, ens_scores, 'Ensemble')

print_classification_report(y_test, ens_pred, 'Ensemble')
plot_confusion_matrix(y_test, ens_pred, labels=[0, 1], title='Ensemble', save_path='../docs/figures/confusion_matrix_ensemble.png')

## 6. Model Comparison

In [ ]:
all_results = [rf_results, xgb_results, ae_results, ens_results]
comparison = compare_models(all_results)
print(comparison)

# Save for dashboard
comparison.to_csv('../docs/figures/model_comparison.csv')

# ROC Curves
plot_roc_curves({
    'Random Forest': (y_test, rf_proba),
    'XGBoost': (y_test, xgb_proba),
    'Autoencoder': (y_test, ae_scores),
    'Ensemble': (y_test, ens_scores),
}, save_path='../docs/figures/roc_curves.png')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
comparison[['accuracy', 'f1', 'precision', 'recall']].plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('Model Comparison — Binary Classification (NSL-KDD)')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../docs/figures/model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_imp = rf.feature_importance(features).head(15)
rf_imp.plot(kind='barh', ax=axes[0], color='#27ae60')
axes[0].set_title('Random Forest — Feature Importance')
axes[0].invert_yaxis()

xgb_imp = xgb.feature_importance(features).head(15)
xgb_imp.plot(kind='barh', ax=axes[1], color='#2980b9')
axes[1].set_title('XGBoost — Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../docs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Model | Accuracy | F1 | AUC |
|-------|----------|----|-----|
| Random Forest | ... | ... | ... |
| XGBoost | ... | ... | ... |
| Autoencoder | ... | ... | ... |
| Ensemble | ... | ... | ... |

**Next**: SHAP explainability analysis (notebook 03)